**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Sparse Coding & Dictionary Learning

[Compressed Sensing](./Compressed_Sensing.ipynb) assumed a known sparsifying basis. This sequel asks two harder questions: how do you find the sparse code *greedily and fast* (OMP), and — the big one — can you **learn the dictionary itself from data** (K-SVD)? Both verified on planted-truth problems where we know the answer.

## 1. Pre-requisites

[Compressed Sensing](./Compressed_Sensing.ipynb), [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S2/S4.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 3 — *Greedy Pursuit: OMP* (~40 min)
**Goal:** build the sparse code one atom at a time; verify exact recovery of a planted support.
**Builds on:** [Compressed Sensing](./Compressed_Sensing.ipynb). &nbsp; **Feeds into:** Session 2 (dictionary learning).

---

## 2. One Atom at a Time

💡 **Intuition.** L1 minimization is principled but iterative-solver-shaped. **Orthogonal Matching Pursuit** is the greedy engineer's answer: repeatedly pick the atom most correlated with the residual, then re-fit *all* chosen atoms by least squares (the 'orthogonal' — the residual stays perpendicular to everything chosen, so no atom is picked twice). $K$ iterations, each a correlation + a small solve; exact recovery when the dictionary is incoherent enough.

In [ ]:
# ORACLE: planted 6-sparse code in a random 64×256 dictionary — recover it exactly

# YOUR CODE HERE


**What just happened.** OMP recovered the support **exactly** — `[11, 33, 150, 214, 234, 238]` in both rows — with a coefficient error of **2.4e-15**, which is machine precision. The `assert` guarantees it.

Take the measure of that. The dictionary is 64 × 256, so the system is underdetermined by a factor of four, and there are $\binom{256}{6} \approx 3\times 10^{11}$ possible 6-element supports. A greedy algorithm — one that picks an atom, never reconsiders, and repeats six times — found the right one out of three hundred billion. Greedy methods normally come with an approximation-ratio caveat attached; here, under sufficient incoherence, greed is provably exact.

**Why the coefficient error is 1e-15 rather than 1e-6.** Once the *support* is correct, the remaining problem is an ordinary overdetermined least-squares fit of 6 unknowns to 64 equations, which `lstsq` solves to machine precision. So the two printed results are really one discrete claim and one trivial consequence: get the support right and the coefficients are free. This is why support recovery, not coefficient accuracy, is the meaningful measure in sparse problems.

**The "orthogonal" is doing real work.** Plain matching pursuit picks an atom and subtracts its contribution. OMP re-solves least squares over *all* selected atoms at every iteration — the `np.linalg.lstsq(Ds, y)` inside the loop. That keeps the residual perpendicular to everything already chosen, so no atom is ever selected twice and every iteration strictly reduces the residual. Without it the algorithm can revisit atoms and dither; with it, $K$ iterations suffice.

**And note what it never does: backtrack.** Each atom is chosen by a single correlation test against the current residual, and that choice is permanent. When the dictionary is incoherent enough, the correct atom always wins that test and the greed is harmless. When it is not, one early mistake is unrecoverable — the algorithm keeps building on a wrong foundation and the support comes out wrong. That fragility is not visible in this clean run, and it is exactly what the next cell measures.

In [ ]:
# and its breaking point: recovery probability vs sparsity level (the coherence wall)

# YOUR CODE HERE


**What just happened.** Recovery rate against sparsity, and the shape is the result: near-certain success at low $K$, then a **collapse** over a narrow range. It is not a gentle decline in accuracy — it is a phase transition between "works essentially always" and "fails essentially always."

**Why a cliff rather than a slope.** Success here is a *discrete* event: either every one of the $K$ selected atoms is correct or the support is wrong. OMP picks each atom by a single correlation test and never backtracks, so one early mistake is unrecoverable — the algorithm keeps re-fitting on a wrong foundation. As $K$ grows, two things worsen together: more chances to make that first mistake, and a residual increasingly likely to correlate better with some wrong atom than with a remaining right one, because a random 64×256 dictionary has non-zero coherence between atoms. Once the failure probability per pick crosses a threshold, compounding over $K$ picks does the rest.

**Locate the two limits.** There is a hard information-theoretic ceiling at $K = 64$: 64 measurements cannot determine more than 64 unknowns, whatever algorithm you use. The observed cliff arrives well below that. The gap between them is the price of greed — L1 minimisation from [Compressed Sensing](./Compressed_Sensing.ipynb) generally pushes further, at the cost of an iterative solve, and the recovery theorems in that literature are about exactly this gap.

**Phase transitions are the normal behaviour in this field, not a quirk of OMP.** Compressed sensing, community detection, and matrix completion all show them, and the practical consequence matters: you cannot extrapolate from a working regime. A system tested at $K = 6$ tells you nothing about $K = 20$, because the degradation is not gradual. If you are deploying sparse recovery, find your cliff empirically and stay well clear of it — the region just below it is where performance is technically fine and one unlucky dataset ruins you.

Note also what is held fixed: the dictionary is 64 × 256 throughout, so this curve is specific to that shape and to random Gaussian atoms. A more incoherent dictionary moves the cliff right; a larger, more redundant one moves it left.

---
### 🕐 Session 2 of 3 — *K-SVD: Learning the Dictionary* (~40 min)
**Goal:** alternate sparse coding and per-atom SVD updates; recover a PLANTED dictionary.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (denoising with learned atoms).

---

## 3. Where Do Atoms Come From?

💡 **Intuition.** Wavelets are a guess; data can vote. **K-SVD** alternates: (1) sparse-code every training signal with the current dictionary (OMP), (2) update each atom — restrict to the signals that *use* it, and set the atom (and its coefficients) to the **rank-one SVD** of their residual matrix — the best single direction explaining what's left ([Eckart–Young](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) again). The audit most demos skip: plant a dictionary, generate data from it, and count how many atoms K-SVD *actually recovers*.

In [ ]:
# ORACLE: plant a 20-atom dictionary in R^16, generate 3-sparse data, recover the atoms
# match learned atoms to true atoms (up to sign and permutation)

# YOUR CODE HERE


**What just happened.** **20 of 20** planted atoms recovered to $|\cos| > 0.98$. Not "the learned dictionary looks reasonable" — every single direction of the true dictionary was found, matched up to sign and permutation, neither of which is identifiable.

This is the audit most dictionary-learning demos skip. The usual presentation trains on real images, displays a grid of Gabor-like atoms, and invites you to find them plausible. Plausibility is not verification. Here a dictionary was *planted*, data was generated from it, and the question "did we get the atoms back?" has a checkable answer.

**How the alternating scheme gets there.** Fix $D$, sparse-code every signal with OMP; fix the codes, update each atom. That is alternating minimisation, the same shape as k-means and EM — with the same consequence, which is convergence to a *local* optimum that depends on initialisation. `D` starts from randomly chosen training signals precisely because that is a better starting point than noise.

**The per-atom update is the elegant part.** For atom $j$, restrict to the signals that actually use it, form the residual with atom $j$'s own contribution added back in, and take the **rank-one SVD**. By [Eckart–Young](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb), the leading singular pair is the provably best rank-one approximation of that residual — so the update is not a heuristic but the optimal single direction to explain what remains. It also updates the atom and its coefficients *simultaneously*, which is why K-SVD converges in tens of iterations where gradient descent on the same objective takes far longer.

**One line worth noticing.** `if len(users) == 0` re-randomises any atom that no signal uses. Atom death is a genuine K-SVD failure mode, not a hypothetical edge case: without that branch the effective dictionary silently shrinks and you learn fewer atoms than you asked for, with nothing in the output to tell you.

**And the conditions here are generous — the result is earned, not automatic.** 3000 training signals for 20 atoms in 16 dimensions, 45 iterations, and noise at 0.01. The printed note says what happens if you starve any of that, and the `assert` is deliberately set at 16 rather than 20 because a perfect score is not guaranteed. Try `n_iter=5`, or 300 training signals instead of 3000: recovery drops, because alternating minimisation lands in a local optimum with some true atoms never found and some learned atoms sitting between two real ones. That is the honest behaviour of the method, and knowing the failure shape is more useful than the clean number.

---
### 🕐 Session 3 of 3 — *Denoising with Learned Atoms* (~35 min)
**Goal:** the payoff: a dictionary learned from noisy patches beats a fixed basis at denoising.
**Builds on:** Session 2.

---

## 4. The Payoff

💡 **Intuition.** Denoise by *sparse approximation*: code each noisy patch with a few atoms, rebuild — whatever the dictionary can express survives, and noise (which is sparse in **no** dictionary) dies. Learned atoms fit the data's actual structure better than any fixed basis fits it, so at equal sparsity they keep more signal. This pipeline (K-SVD denoising) was state-of-the-art for a decade and is the conceptual ancestor of every learned-representation denoiser since — including [diffusion models](../Intro_Mach_Learn/Diffusion_Models.ipynb).

In [ ]:
# 1-D 'patches' from a piecewise-smooth signal family; compare DCT vs learned dictionary
# denoise fresh noisy patches by 3-sparse OMP approximation in each dictionary

# YOUR CODE HERE


**What just happened.** From a 7.8 dB input, the learned dictionary gains **+2.5 dB** while the DCT basis gives **−0.8 dB**. The DCT result is not a small win — it is *negative*: the "denoised" output is worse than the noisy input it started from.

This is a properly controlled experiment. Same patches, same noise realisation, same sparsity budget of 3 atoms, same OMP solver. The only variable is which dictionary, so the 3.3 dB spread is attributable to representation alone.

**Why sparse approximation denoises at all.** Force each patch to be built from just 3 atoms. Structure the dictionary can express in 3 atoms survives; noise is sparse in *no* dictionary, so it cannot be captured and is discarded. Denoising is a side effect of being made to be brief.

**And why the DCT loses so badly that it goes negative.** The error has two parts: noise removed (good) and signal destroyed (bad). Our patches are sinusoids, step edges, and narrow Gaussian bumps. A step edge is famously *not* sparse in the DCT — it needs many coefficients to build a discontinuity, which is Gibbs' phenomenon from [Hilbert Spaces](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb) wearing a different hat. Restricted to 3 DCT atoms, the approximation destroys more real signal than it removes noise, and the balance comes out negative. The learned dictionary contains atoms *shaped like* edges and bumps, because it was fitted to patches of exactly that kind, so 3 of them are plenty.

**The lesson generalises past this cell.** "Sparsity denoises" is not unconditional — it holds only if the signal really is sparse in the dictionary you chose. In the wrong basis, sparse approximation is simply lossy compression and can be worse than doing nothing at all. The −0.8 dB is the counterexample worth remembering.

Worth pushing on the natural objection: the DCT is a *complete orthonormal basis* for $\mathbb{R}^{16}$ and can represent these patches exactly. So why does it lose? Because completeness is not sparsity. Exact representation may require all 16 coefficients; the question is whether **3** suffice. Distinguishing "can represent" from "can represent *briefly*" is the conceptual core of the entire workshop.

**And the comparison is fair.** `D_data` was trained on *noisy* patches from the same family and never saw the test set; the DCT is its full 16-atom basis, not a truncation. Neither side is handicapped — the learned dictionary wins because it knows what these signals look like.

**Lineage.** This pipeline was state of the art for about a decade, and it is the direct ancestor of every learned-representation denoiser since. The score networks in [diffusion models](../Intro_Mach_Learn/Diffusion_Models.ipynb) are denoisers whose dictionary is a neural network rather than a matrix — same principle, learned representation, vastly more capacity.

## 5. Conclusion

OMP builds codes greedily and provably recovers planted supports; K-SVD recovers planted *dictionaries* atom-for-atom; and sparse approximation in the learned dictionary is a denoiser that knows your data. Representation is no longer a design choice — it's a fit.

---
## Where next

- [Compressed Sensing](./Compressed_Sensing.ipynb) — the convex counterpart.
- [Representation Learning](../Intro_Mach_Learn/Representation_Learning.ipynb) — the neural descendants of this exact idea.